# L4 · Análise de Séries Temporais para Executivos

**Análise Prescritiva — Camada de Alfabetização de Dados (Learning)**

| Campo | Detalhe |
|---|---|
| **Notebook** | L4 · Time Series Analysis for Executives |
| **Autor** | Matheus Mendes |
| **Data** | 27/julho/2026 |
| **Versão** | 1.0 |
| **Público-alvo** | Executivos não-técnicos — "para onde isso vai, e dá pra confiar?" |
| **Dependências** | numpy, pandas, plotly, statsmodels |

---

## Por que este notebook existe

Em **L0** você aprendeu a **ler números** (média, variância, percentil, correlação).
Em **L1** aprendeu a falar de **incerteza** (probabilidade, valor esperado, Bayes).
Em **L3** aprendeu a transformar correlação em **previsão** (regressão linear).

Em **L4** entra a dimensão que faltava: o **tempo**. Quase todo número que
importa num board — câmbio PTAX, vendas mensais, custo do BOM, market share —
não é um ponto isolado: é uma **série que se move mês a mês**. E séries no
tempo têm regras próprias que a estatística de L0 ignora.

A pergunta executiva de L4 não é "quanto?" nem "qual a chance?" nem "se eu mexo
aqui, o que acontece?". É:

> **"Para onde este número está indo — e o quanto posso confiar na projeção?"**

Este material segue o formato **narrativa-primeiro**:

> **Conceito → Intuição → Matemática → Código → Recado Executivo**

Todos os números vêm do **case BYD Camaçari**: a série mensal do **PTAX**
(R$/US$, 2021–2026) e as **vendas mensais** do portfólio BYD no Brasil.
O tema escuro (`#0d1117`) e a paleta (`azul-petróleo · vermelho-tijolo · teal ·
violeta · laranja-âmbar`) seguem o mesmo padrão visual dos cadernos anteriores.

---

### Os 6 conceitos deste caderno

| # | Conceito | Pergunta que responde | Exemplo BYD |
|---|----------|-----------------------|-------------|
| 1 | **Série temporal** | O que muda no tempo? | PTAX mês a mês, 2021–2026 |
| 2 | **Tendência (*trend*)** | Para onde está indo? | PTAX subindo no longo prazo |
| 3 | **Sazonalidade** | Que padrão se repete? | Vendas fortes em Q4/Q1 |
| 4 | **Estacionariedade** | É estável ou está à deriva? | Nível vs. variação do PTAX |
| 5 | **Autocorrelação** | O que prevê a si mesmo? | Vendas de hoje ecoam há 12 meses |
| 6 | **Previsão (*forecast*)** | O que vem pela frente? | Vendas BYD 12 meses à frente |

Cada seção fecha com um **Recado Executivo** — a frase que você leva para a
reunião.


In [1]:
# ──────────────────────────────────────────────────────────────
# Setup — tema escuro, paleta validada, séries do case BYD
# ──────────────────────────────────────────────────────────────
import json
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "notebook_connected"
np.random.seed(42)

# ── Diretórios de saída (JSON + HTMLs das visualizações) ──
NOTEBOOK_ROOT = Path(r"C:\Users\mathe\code_space\orchestration\value-factory\case-studies\byd-camacari-2025-2027\analise-prescritiva")
OUT_DIR  = NOTEBOOK_ROOT / "outputs" / "learning"
HTML_DIR = OUT_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Paleta validada no dataviz skill (ALL CHECKS PASS, surface #0d1117) ──
BG      = "#0d1117"
INK     = "#e8edf5"
MUTED   = "#9baabb"
GRID    = "#30363d"

AZUL    = "#0284c7"
TIJOLO  = "#dc2626"
TEAL    = "#0d9488"
VIOLETA = "#9333ea"
AMBAR   = "#ea580c"

STATUS_GOOD = "#22c55e"
STATUS_WARN = AMBAR
STATUS_BAD  = TIJOLO

def style(fig, title, height=480):
    fig.update_layout(
        template="plotly_dark", paper_bgcolor=BG, plot_bgcolor=BG,
        title=dict(text=title, x=0.5, xanchor="center",
                   font=dict(size=17, color=INK)),
        font=dict(color=INK, size=13),
        legend=dict(font=dict(color=MUTED), bgcolor="rgba(13,17,23,0.6)",
                    bordercolor=GRID, borderwidth=1),
        margin=dict(l=60, r=40, t=70, b=55), height=height,
    )
    fig.update_xaxes(color=MUTED, gridcolor=GRID, zerolinecolor=GRID, linecolor=GRID)
    fig.update_yaxes(color=MUTED, gridcolor=GRID, zerolinecolor=GRID, linecolor=GRID)
    return fig

# ── Série 1: PTAX mensal (R$/US$) calibrada, jan/2021 -> jun/2026 ──
# Passeio aleatório com deriva (random walk + drift): o NÍVEL é
# não-estacionário (vagueia, sem âncora — ADF não rejeita), mas a
# VARIAÇÃO mensal é estacionária. Termina em 5.1176 (PTAX atual).
meses = pd.date_range("2021-01-01", "2026-06-01", freq="MS")
n = len(meses)
t = np.arange(n)
drift   = 0.016                                    # deriva de alta por mês
choques = np.random.normal(drift, 0.05, n)         # variação mensal (estacionária)
ptax = 5.10 + np.cumsum(choques)                   # nível = soma acumulada (passeio)
ptax = ptax - ptax[-1] + 5.1176                    # ancora o último no PTAX atual
ptax = np.round(np.clip(ptax, 4.4, 6.4), 4)
ptax_s = pd.Series(ptax, index=meses, name="PTAX")

# ── Série 2: vendas mensais BYD Brasil (unidades), com tendência + sazonalidade ──
base_vendas = 1200 + 42 * t                          # rampa de ramp-up BYD BR
saz_mensal = np.array([1.18, 1.10, 1.02, 0.95, 0.90, 0.86,   # jan..jun
                        0.88, 0.94, 1.00, 1.06, 1.14, 1.22])  # jul..dez
saz = np.array([saz_mensal[m.month - 1] for m in meses])
ruido = np.random.normal(0, 0.05, n)
vendas = base_vendas * saz * (1 + ruido)
vendas = np.round(vendas).astype(int)
vendas_s = pd.Series(vendas, index=meses, name="vendas")

BYD = {
    "ptax_atual":          float(ptax_s.iloc[-1]),
    "ptax_medio_periodo":  round(float(ptax_s.mean()), 4),
    "vol_anual_pct":       14.2,
    "imported_share_bom":  0.42,
    "incentivo_coverage":  0.18,
    "vendas_ult_mes":      int(vendas_s.iloc[-1]),
    "vendas_media_2026":   int(vendas_s["2026"].mean()),
    "meses_serie":         int(n),
}

print("Setup OK · tema escuro carregado · 2 séries BYD prontas")
print(f"PTAX: {n} meses ({meses[0]:%b/%Y} -> {meses[-1]:%b/%Y}) · "
      f"atual R$ {BYD['ptax_atual']:.4f}")
print(f"Vendas BYD BR: ultimo mes {BYD['vendas_ult_mes']} un · "
      f"media 2026 {BYD['vendas_media_2026']} un/mes")


Setup OK · tema escuro carregado · 2 séries BYD prontas
PTAX: 66 meses (Jan/2021 -> Jun/2026) · atual R$ 5.1176
Vendas BYD BR: ultimo mes 3391 un · media 2026 3712 un/mes


---

## 1 · Série temporal — o número que carrega sua própria história

### ① Por que isto importa
Um relatório mostra "PTAX = R$ 5,12". Ok — mas **5,12 vindo de onde?** Se veio
caindo de 6,20, a leitura é "alívio, hedge relaxa". Se veio subindo de 4,60, é
"alerta, custo do BOM disparando". **O mesmo número significa coisas opostas
dependendo do caminho.** Série temporal é o número **com o caminho anexado** —
e sem o caminho, decisão nenhuma sobre câmbio, vendas ou estoque faz sentido.

### ② Conceito, sem jargão
Série temporal = uma sequência de medições do **mesmo indicador**, tomadas em
**intervalos regulares** (todo mês, todo dia), **na ordem em que aconteceram**.
A ordem é sagrada: embaralhar uma série temporal destrói a informação, ao
contrário de uma amostra de L0 (onde a ordem não importa).

### ③ Intuição — o PTAX mês a mês
A figura abaixo é a série do PTAX (R$/US$) de jan/2021 a jun/2026. Repare que
ela **não pula aleatoriamente**: cada mês fica *perto* do anterior (o dólar não
salta de 5 para 8 de um mês pro outro) e há trechos de alta e de calmaria. Essa
"memória de curto prazo" é a assinatura de toda série temporal.

### ④ A matemática
Formalmente uma série é $\{y_t\}_{t=1}^{T}$, indexada pelo tempo $t$. Quase
sempre a decompomos em três ingredientes:

$$y_t = \underbrace{T_t}_{\text{tendência}} + \underbrace{S_t}_{\text{sazonalidade}} + \underbrace{R_t}_{\text{resíduo}}$$

As três próximas seções (Tendência, Sazonalidade, Estacionariedade) são
exatamente sobre **isolar cada uma dessas parcelas**.


In [2]:
# ──────────────────────────────────────────────────────────────
# Série temporal — o PTAX mês a mês (a "história" do câmbio)
# ──────────────────────────────────────────────────────────────
ult = ptax_s.iloc[-1]
pico = ptax_s.max();  vale = ptax_s.min()
mes_pico = ptax_s.idxmax(); mes_vale = ptax_s.idxmin()
amplitude_pct = (pico - vale) / vale * 100

print(f"PTAX periodo: {ptax_s.index[0]:%b/%Y} -> {ptax_s.index[-1]:%b/%Y}")
print(f"  atual : R$ {ult:.4f}")
print(f"  pico  : R$ {pico:.4f} ({mes_pico:%b/%Y})")
print(f"  vale  : R$ {vale:.4f} ({mes_vale:%b/%Y})")
print(f"  amplitude pico->vale: {amplitude_pct:.1f}% — mesmo numero, historias opostas")

fig = go.Figure()
fig.add_scatter(x=ptax_s.index, y=ptax_s.values, mode="lines",
                line=dict(color=AZUL, width=2.5), name="PTAX (R$/US$)",
                hovertemplate="%{x|%b/%Y}<br>R$ %{y:.4f}<extra></extra>")
fig.add_scatter(x=[mes_pico], y=[pico], mode="markers+text", text=["pico"],
                textposition="top center", textfont=dict(color=TIJOLO),
                marker=dict(color=TIJOLO, size=11), name="pico",
                hovertemplate="pico R$ %{y:.4f}<extra></extra>")
fig.add_scatter(x=[mes_vale], y=[vale], mode="markers+text", text=["vale"],
                textposition="bottom center", textfont=dict(color=STATUS_GOOD),
                marker=dict(color=STATUS_GOOD, size=11), name="vale",
                hovertemplate="vale R$ %{y:.4f}<extra></extra>")
fig.add_scatter(x=[ptax_s.index[-1]], y=[ult], mode="markers",
                marker=dict(color=INK, size=12, symbol="star"), name="atual")
style(fig, "1 · Série temporal do PTAX — o número com o caminho anexado")
fig.update_yaxes(title="R$ por US$")
fig.write_html(str(HTML_DIR / "l4-01-serie-ptax.html"),
               include_plotlyjs="cdn", full_html=True)
fig.show()


PTAX periodo: Jan/2021 -> Jun/2026
  atual : R$ 5.1176
  pico  : R$ 5.1176 (Jun/2026)
  vale  : R$ 4.6063 (Jan/2021)
  amplitude pico->vale: 11.1% — mesmo numero, historias opostas


### ⑤ Recado executivo — Série temporal
> - **Todo indicador de board é uma série, não um ponto.** "PTAX = 5,12" sem o
>   caminho é meia-informação: 5,12 *subindo* e 5,12 *caindo* pedem decisões opostas.
> - **A ordem carrega a informação.** Diferente de uma amostra estática (L0),
>   embaralhar uma série destrói tudo — cada mês herda o anterior.
> - **Toda série se decompõe em tendência + sazonalidade + resíduo.** Saber qual
>   parcela está mandando é o que separa "reação a ruído" de "reação a sinal".


---

## 2 · Tendência — para onde a série está *realmente* indo

### ① Por que isto importa
O ruído mês a mês engana. Um executivo vê o PTAX cair de 5,30 para 5,18 e
comemora — mas se a **tendência** de fundo é de alta, essa queda é só um soluço
antes de novos recordes. Tendência é o **rumo estrutural**, filtrando o
chiado. Confundir soluço com virada de rumo é a origem #1 de hedge desmontado
cedo demais.

### ② Conceito, sem jargão
Tendência = o movimento **lento e persistente** que sobrevive quando você
"borra" o zigue-zague de curto prazo. A ferramenta clássica de borrar é a
**média móvel**: em vez do valor do mês, olhe a média dos últimos 12 meses —
o vaivém se cancela e sobra o rumo.

### ③ Intuição — média móvel de 12 meses do PTAX
A linha fina (azul) é o PTAX cru; a linha grossa (âmbar) é a média móvel de 12
meses. A grossa ignora os soluços e mostra o **rumo**: onde ela inclina para
cima, o câmbio está estruturalmente se depreciando — custo do BOM importado
(42%) sob pressão persistente, não pontual.

### ④ A matemática
Média móvel centrada de janela $k$:

$$\text{MM}_t = \frac{1}{k}\sum_{i=-k/2}^{k/2} y_{t+i}$$

Alternativa paramétrica: ajustar uma reta $y_t = \beta_0 + \beta_1 t$ (regressão
de L3 com o tempo como X). O **sinal de $\beta_1$** é o rumo; sua **magnitude**
é a velocidade da deriva (R$/mês).


In [3]:
# ──────────────────────────────────────────────────────────────
# Tendência — média móvel de 12m + reta de tendência (β1 = rumo)
# ──────────────────────────────────────────────────────────────
mm12 = ptax_s.rolling(12, center=True).mean()

# reta de tendência (regressão de L3 com o tempo como X)
tt = np.arange(len(ptax_s))
beta1, beta0 = np.polyfit(tt, ptax_s.values, 1)
reta = beta0 + beta1 * tt
deriva_ano = beta1 * 12

print(f"Reta de tendência: PTAX = {beta0:.3f} + ({beta1:+.4f}) * mes")
print(f"  slope β1 = {beta1:+.4f} R$/mes  ->  deriva de {deriva_ano:+.3f} R$/ano")
print(f"  rumo: {'ALTA (depreciacao do real)' if beta1 > 0 else 'BAIXA'} — "
      f"tendencia estrutural, nao soluco mensal")

fig = go.Figure()
fig.add_scatter(x=ptax_s.index, y=ptax_s.values, mode="lines",
                line=dict(color=AZUL, width=1.2), opacity=0.55, name="PTAX cru",
                hovertemplate="%{x|%b/%Y}<br>R$ %{y:.4f}<extra></extra>")
fig.add_scatter(x=mm12.index, y=mm12.values, mode="lines",
                line=dict(color=AMBAR, width=4), name="Média móvel 12m (rumo)",
                hovertemplate="MM12 %{x|%b/%Y}<br>R$ %{y:.4f}<extra></extra>")
fig.add_scatter(x=ptax_s.index, y=reta, mode="lines",
                line=dict(color=VIOLETA, width=2, dash="dash"),
                name=f"Reta tendência · {beta1:+.4f} R$/mes")
style(fig, "2 · Tendência do PTAX — média móvel filtra o soluço, revela o rumo")
fig.update_yaxes(title="R$ por US$")
fig.write_html(str(HTML_DIR / "l4-02-tendencia.html"),
               include_plotlyjs="cdn", full_html=True)
fig.show()


Reta de tendência: PTAX = 4.743 + (+0.0032) * mes
  slope β1 = +0.0032 R$/mes  ->  deriva de +0.039 R$/ano
  rumo: ALTA (depreciacao do real) — tendencia estrutural, nao soluco mensal


### ⑤ Recado executivo — Tendência
> - **Tendência é rumo, não o último ponto.** A média móvel de 12 meses borra o
>   ruído e mostra se o câmbio está estruturalmente subindo — o que decide hedge,
>   não a queda pontual do mês.
> - **Um número resume o rumo: o slope ($\beta_1$).** Sinal = direção;
>   magnitude = velocidade da deriva (R$/ano). É a mesma regressão de L3, com o
>   tempo no eixo X.
> - **Não confunda soluço com virada.** Reagir a cada zigue-zague desmonta
>   proteção cedo demais; reaja quando a *média móvel* mudar de inclinação.


---

## 3 · Sazonalidade — o padrão que volta todo ano no mesmo mês

### ① Por que isto importa
Vendas de dezembro sempre batem julho. Se o board compara **dezembro contra
novembro** e comemora "+15%!", pode estar celebrando pura sazonalidade — o
mesmo pulo acontece **todo ano**, sem mérito de gestão nenhum. Ignorar
sazonalidade produz metas erradas, estoque errado e bônus pago por sorte de
calendário.

### ② Conceito, sem jargão
Sazonalidade = um padrão que **se repete em ciclo fixo** (normalmente 12 meses).
Não é tendência (que não volta) nem ruído (que não tem padrão): é o **ritmo do
calendário**. A regra de ouro do executivo: **compare mês contra o mesmo mês do
ano anterior (YoY)**, nunca contra o mês vizinho.

### ③ Intuição — o perfil sazonal das vendas BYD
O gráfico mostra o **fator sazonal médio de cada mês** das vendas BYD Brasil.
Valores acima de 1,0 = meses estruturalmente fortes (Q4/Q1, comprador antecipa
IPVA e usa 13º); abaixo de 1,0 = vale do meio do ano. É o mesmo padrão todo ano
— planeje produção de Camaçari em cima dele.

### ④ A matemática
Isolamos a sazonalidade **destendenciando** e tirando a média por mês do ano:

$$S_m = \frac{1}{N_m}\sum_{t:\,\text{mês}(t)=m} \frac{y_t}{\text{MM}_t}$$

O fator $S_m$ (dez ≈ 1,22, jun ≈ 0,86) multiplica ou desconta a tendência.
Dessazonalizar = dividir cada mês pelo seu $S_m$, revelando o movimento "limpo".


In [4]:
# ──────────────────────────────────────────────────────────────
# Sazonalidade — fator médio por mês do ano (vendas BYD BR)
# ──────────────────────────────────────────────────────────────
mm_v = vendas_s.rolling(12, center=True).mean()
detrend = (vendas_s / mm_v).dropna()
fator = detrend.groupby(detrend.index.month).mean()
fator = fator / fator.mean()           # normaliza para media 1.0
nomes = ["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"]

mes_forte = int(fator.idxmax()); mes_fraco = int(fator.idxmin())
spread = (fator.max() / fator.min() - 1) * 100
print("Fator sazonal por mês (1.0 = média):")
for m in range(1, 13):
    print(f"  {nomes[m-1]}: {fator.get(m, float('nan')):.3f}")
print(f"\nMes mais forte: {nomes[mes_forte-1]} ({fator.max():.2f}) · "
      f"mais fraco: {nomes[mes_fraco-1]} ({fator.min():.2f})")
print(f"Spread sazonal: {spread:.0f}% — compare YoY, nunca contra o mes vizinho")

cores = [STATUS_GOOD if fator[m] >= 1 else AMBAR for m in range(1, 13)]
fig = go.Figure()
fig.add_bar(x=nomes, y=[fator[m] for m in range(1, 13)],
            marker=dict(color=cores, line=dict(color=INK, width=1)),
            hovertemplate="%{x}<br>fator %{y:.3f}<extra></extra>", name="fator sazonal")
fig.add_hline(y=1.0, line=dict(color=MUTED, width=2, dash="dash"),
              annotation_text="média (1.0)", annotation_font_color=MUTED)
style(fig, "3 · Sazonalidade das vendas BYD BR — o ritmo do calendário")
fig.update_yaxes(title="fator sazonal (× tendência)", range=[0.7, 1.35])
fig.write_html(str(HTML_DIR / "l4-03-sazonalidade.html"),
               include_plotlyjs="cdn", full_html=True)
fig.show()


Fator sazonal por mês (1.0 = média):
  Jan: 1.143
  Fev: 1.062
  Mar: 0.993
  Abr: 0.927
  Mai: 0.932
  Jun: 0.834
  Jul: 0.863
  Ago: 0.919
  Set: 0.953
  Out: 1.028
  Nov: 1.099
  Dez: 1.246

Mes mais forte: Dez (1.25) · mais fraco: Jun (0.83)
Spread sazonal: 49% — compare YoY, nunca contra o mes vizinho


### ⑤ Recado executivo — Sazonalidade
> - **Dezembro sempre bate julho — não é mérito, é calendário.** O fator sazonal
>   (dez ≈ 1,2; jun ≈ 0,86) se repete todo ano; celebrar +15% mês-contra-mês é
>   celebrar o relógio.
> - **Regra de ouro: compare YoY (mesmo mês, ano anterior).** É a única forma de
>   separar gestão de sazonalidade.
> - **Sazonalidade é planejável.** Camaçari deve produzir em cima do perfil
>   sazonal — estoque em out/nov para o pico de dez/jan, não reação tardia.


---

## 4 · Estacionariedade — estável (previsível) vs. à deriva (traiçoeiro)

### ① Por que isto importa
Quase todo modelo de previsão **exige** que a série seja estável — média e
variância que não mudam com o tempo. O PTAX em **nível** não é: ele *passeia*,
sem âncora, e "o câmbio médio dos últimos 5 anos" não prevê nada. Já a
**variação mensal** do PTAX é estável. Rodar previsão numa série à deriva
produz o número bonito e falso que estoura o orçamento seis meses depois.

### ② Conceito, sem jargão
**Estacionária** = a série "esquece" onde está; oscila em torno de uma média
fixa, sem deriva (ex.: retornos, variações). **Não-estacionária** = a série tem
memória longa e vagueia para longe (ex.: o nível do PTAX, o nível de vendas em
ramp-up). O truque padrão para *estabilizar* uma série que vagueia: olhar a
**diferença** entre meses consecutivos ($y_t - y_{t-1}$) em vez do nível.

### ③ Intuição — nível (à deriva) vs. variação (estável)
Painel esquerdo: o **nível** do PTAX — a média local muda o tempo todo, sem
âncora (não-estacionária). Painel direito: a **variação mensal** — oscila em
torno de zero com amplitude constante (estacionária). Só a da direita é
material honesto para previsão.

### ④ A matemática
O teste padrão é o **ADF (Augmented Dickey-Fuller)**. Hipótese nula: "a série
tem raiz unitária" (= é não-estacionária, vagueia). Regra de bolso executiva:

$$p\text{-valor} < 0{,}05 \;\Rightarrow\; \text{estacionária (pode modelar)}$$
$$p\text{-valor} \ge 0{,}05 \;\Rightarrow\; \text{diferencie antes de modelar}$$


In [5]:
# ──────────────────────────────────────────────────────────────
# Estacionariedade — nível (à deriva) vs. diferença (estável) + ADF
# ──────────────────────────────────────────────────────────────
from statsmodels.tsa.stattools import adfuller

nivel = ptax_s.values
dif   = ptax_s.diff().dropna()

adf_nivel = adfuller(nivel, autolag="AIC")
adf_dif   = adfuller(dif.values, autolag="AIC")
p_nivel, p_dif = adf_nivel[1], adf_dif[1]

def veredito(p):
    return "ESTACIONARIA (pode modelar)" if p < 0.05 else "NAO-estacionaria (diferencie)"

print("Teste ADF (Augmented Dickey-Fuller):")
print(f"  PTAX em NIVEL      : p-valor = {p_nivel:.3f}  ->  {veredito(p_nivel)}")
print(f"  PTAX em VARIACAO   : p-valor = {p_dif:.3f}  ->  {veredito(p_dif)}")
print(f"  media do nivel move; media da variacao ~ {dif.mean():+.4f} (ancorada em zero)")

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=(f"Nivel — a deriva (ADF p={p_nivel:.2f})",
                                    f"Variacao mensal — estavel (ADF p={p_dif:.2f})"))
fig.add_scatter(x=ptax_s.index, y=nivel, mode="lines",
                line=dict(color=TIJOLO, width=2), name="nivel PTAX", row=1, col=1)
fig.add_scatter(x=dif.index, y=dif.values, mode="lines",
                line=dict(color=TEAL, width=1.8), name="delta PTAX (mes)", row=1, col=2)
fig.add_hline(y=float(dif.mean()), line=dict(color=INK, width=2, dash="dash"),
              row=1, col=2)
style(fig, "4 · Estacionariedade — so a serie estavel (direita) preve honesto",
      height=430)
for ann in fig.layout.annotations:
    ann.font.color = INK; ann.font.size = 13
fig.write_html(str(HTML_DIR / "l4-04-estacionariedade.html"),
               include_plotlyjs="cdn", full_html=True)
fig.show()


Teste ADF (Augmented Dickey-Fuller):
  PTAX em NIVEL      : p-valor = 0.461  ->  NAO-estacionaria (diferencie)
  PTAX em VARIACAO   : p-valor = 0.000  ->  ESTACIONARIA (pode modelar)
  media do nivel move; media da variacao ~ +0.0079 (ancorada em zero)


### ⑤ Recado executivo — Estacionariedade
> - **"Câmbio médio dos últimos 5 anos" não prevê nada.** O nível do PTAX
>   *passeia* (não-estacionário, ADF p ≈ alto) — não tem âncora para reverter.
> - **A variação mensal, sim, é estável.** Diferenciar (olhar Δ em vez de nível)
>   é o truque padrão para transformar uma série traiçoeira em modelável.
> - **Pergunte sempre: "esse modelo rodou em série estável?"** Previsão em série
>   à deriva é o número bonito que estoura o orçamento seis meses depois.


---

## 5 · Autocorrelação — o quanto a série prevê a si mesma

### ① Por que isto importa
A pergunta que decide se dá para prever: **"o passado recente carrega
informação sobre o próximo mês?"** Se as vendas de hoje "ecoam" as de 12 meses
atrás, existe estrutura a explorar — e um modelo simples já acerta muito. Se
não há eco nenhum (cada mês é sorteio independente), previsão é chute com
verniz. Autocorrelação **mede esse eco**.

### ② Conceito, sem jargão
Autocorrelação = a correlação de L0, mas da série **com ela mesma defasada**.
"As vendas deste mês se parecem com as de 1 mês atrás? E 12 meses atrás?" Um
pico de autocorrelação no **lag 12** é a assinatura inconfundível de
**sazonalidade anual** — dezembro conversa com o dezembro anterior.

### ③ Intuição — o correlograma (ACF) das vendas BYD
Cada barra é a correlação das vendas com elas mesmas $k$ meses atrás. Barras
altas nos primeiros lags = **memória de curto prazo** (inércia). O pico
destacado no **lag 12** confirma a sazonalidade anual da seção 3. Barras dentro
da faixa cinza (±2/√T) são indistinguíveis de zero — ruído.

### ④ A matemática
Autocorrelação no lag $k$:

$$\rho_k = \frac{\sum_{t=k+1}^{T}(y_t-\bar y)(y_{t-k}-\bar y)}{\sum_{t=1}^{T}(y_t-\bar y)^2}$$

Banda de significância aproximada: $\pm \frac{2}{\sqrt{T}}$. Barras fora da
banda = eco real; dentro = ruído.


In [6]:
# ──────────────────────────────────────────────────────────────
# Autocorrelacao — correlograma (ACF) das vendas BYD BR
# ──────────────────────────────────────────────────────────────
from statsmodels.tsa.stattools import acf

nlags = 18
acf_v = acf(vendas_s.values, nlags=nlags, fft=False)
T = len(vendas_s)
banda_ruido = 2 / np.sqrt(T)
lags = np.arange(nlags + 1)

pico12 = acf_v[12]
signif = [k for k in range(1, nlags + 1) if abs(acf_v[k]) > banda_ruido]
print(f"ACF das vendas BYD (T={T} meses, banda +-{banda_ruido:.2f}):")
print(f"  lag 1  = {acf_v[1]:+.2f}  (inercia de curto prazo)")
print(f"  lag 12 = {pico12:+.2f}  (assinatura de SAZONALIDADE anual)")
print(f"  lags significativos (fora da banda): {signif}")

cores = []
for k in lags:
    if k == 0:
        cores.append(MUTED)
    elif k == 12:
        cores.append(AMBAR)
    elif abs(acf_v[k]) > banda_ruido:
        cores.append(AZUL)
    else:
        cores.append(GRID)

fig = go.Figure()
fig.add_bar(x=lags, y=acf_v, marker=dict(color=cores, line=dict(color=INK, width=0.6)),
            hovertemplate="lag %{x}<br>rho = %{y:.2f}<extra></extra>", name="ACF")
fig.add_hrect(y0=-banda_ruido, y1=banda_ruido, fillcolor="rgba(155,170,187,0.12)",
              line_width=0, annotation_text="faixa de ruido (+-2/raizT)",
              annotation_font_color=MUTED)
fig.add_annotation(x=12, y=pico12, text="lag 12 = sazonalidade",
                   showarrow=True, arrowcolor=AMBAR, font=dict(color=AMBAR),
                   ay=-40)
style(fig, "5 · Autocorrelacao (ACF) das vendas BYD — pico no lag 12 = sazonal")
fig.update_xaxes(title="lag (meses)", dtick=1)
fig.update_yaxes(title="autocorrelacao rho", range=[-0.6, 1.05])
fig.write_html(str(HTML_DIR / "l4-05-autocorrelacao.html"),
               include_plotlyjs="cdn", full_html=True)
fig.show()


ACF das vendas BYD (T=66 meses, banda +-0.25):
  lag 1  = +0.93  (inercia de curto prazo)
  lag 12 = +0.50  (assinatura de SAZONALIDADE anual)
  lags significativos (fora da banda): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]


### ⑤ Recado executivo — Autocorrelação
> - **Autocorrelação responde "dá pra prever?".** É a correlação de L0 da série
>   com o próprio passado: se há eco, há estrutura; se não há, previsão é chute.
> - **Pico no lag 12 = sazonalidade anual comprovada.** O correlograma confirma
>   numericamente o que a seção 3 mostrou no calendário.
> - **Barras dentro da faixa ±2/√T são ruído.** Não construa narrativa em cima
>   de eco que é estatisticamente indistinguível de zero.


---

## 6 · Previsão (*forecast*) — olhar à frente com um intervalo honesto

### ① Por que isto importa
Toda a série existe para responder **uma** pergunta: *o que vem pela frente?*
Mas a previsão que só entrega **um número** ("venderemos 3.400 em dez") é
perigosa — esconde a incerteza. A previsão útil entrega uma **faixa**: "entre
3.050 e 3.750, com 80% de confiança". A largura da faixa é a informação mais
honesta do gráfico — ela diz *quanto* confiar.

### ② Conceito, sem jargão
Forecast = estender tendência + sazonalidade para os próximos meses, somando a
incerteza acumulada. Como a incerteza **cresce com o horizonte** (prever o mês
que vem é fácil; prever daqui a um ano, não), a faixa **abre como um funil**
para a frente. Um método clássico é o **Holt-Winters** (suavização
exponencial), que aprende nível, tendência e sazonalidade e os projeta.

### ③ Intuição — vendas BYD 12 meses à frente
A linha sólida é o histórico; a tracejada âmbar é a previsão de 12 meses; a
área sombreada é o **intervalo de 80%**. Note o funil abrindo: dez/2026 tem
faixa estreita; jun/2027 tem faixa larga. **Planeje capacidade de Camaçari pela
faixa, não pela linha central.**

### ④ A matemática
Holt-Winters aditivo projeta $\hat y_{t+h} = \ell_t + h\,b_t + s_{t+h-m}$
(nível + tendência × horizonte + sazonal). O intervalo aproximado:

$$\hat y_{t+h} \pm z\,\hat\sigma\,\sqrt{h}$$

onde $\hat\sigma$ é o desvio dos resíduos e o $\sqrt{h}$ é o que faz o funil
abrir com o horizonte.


In [7]:
# ──────────────────────────────────────────────────────────────
# Previsao — Holt-Winters (nivel+tendencia+sazonal) 12m a frente
# ──────────────────────────────────────────────────────────────
from statsmodels.tsa.holtwinters import ExponentialSmoothing

modelo = ExponentialSmoothing(
    vendas_s.astype(float), trend="add", seasonal="add",
    seasonal_periods=12, initialization_method="estimated"
).fit()

H = 12
fc = modelo.forecast(H)
resid_sigma = float(np.std(modelo.resid, ddof=1))
z80 = 1.2816
h = np.arange(1, H + 1)
banda_fc = z80 * resid_sigma * np.sqrt(h)
lo, hi = fc.values - banda_fc, fc.values + banda_fc

idx_fc = pd.date_range(vendas_s.index[-1] + pd.offsets.MonthBegin(1),
                       periods=H, freq="MS")

print(f"Previsao vendas BYD BR — {H} meses (Holt-Winters aditivo):")
print(f"  {idx_fc[0]:%b/%Y}: {fc.iloc[0]:,.0f} un  [{lo[0]:,.0f} .. {hi[0]:,.0f}]  (faixa +-{banda_fc[0]:,.0f})")
print(f"  {idx_fc[-1]:%b/%Y}: {fc.iloc[-1]:,.0f} un  [{lo[-1]:,.0f} .. {hi[-1]:,.0f}]  (faixa +-{banda_fc[-1]:,.0f})")
print(f"  funil abre {banda_fc[-1]/banda_fc[0]:.1f}x do 1o ao 12o mes — incerteza cresce com o horizonte")

fig = go.Figure()
fig.add_scatter(x=vendas_s.index, y=vendas_s.values, mode="lines",
                line=dict(color=AZUL, width=2), name="historico",
                hovertemplate="%{x|%b/%Y}<br>%{y:,} un<extra></extra>")
fig.add_scatter(x=list(idx_fc) + list(idx_fc[::-1]),
                y=list(hi) + list(lo[::-1]), fill="toself",
                fillcolor="rgba(234,88,12,0.18)", line=dict(width=0),
                name="intervalo 80%", hoverinfo="skip")
fig.add_scatter(x=idx_fc, y=fc.values, mode="lines+markers",
                line=dict(color=AMBAR, width=3, dash="dash"),
                marker=dict(size=6), name="previsao 12m",
                hovertemplate="%{x|%b/%Y}<br>%{y:,.0f} un<extra></extra>")
fig.add_vline(x=vendas_s.index[-1], line=dict(color=MUTED, width=1.5, dash="dot"),
              annotation_text="hoje", annotation_font_color=MUTED)
style(fig, "6 · Previsao de vendas BYD BR — planeje pela faixa, nao pela linha")
fig.update_yaxes(title="unidades/mes")
fig.write_html(str(HTML_DIR / "l4-06-forecast.html"),
               include_plotlyjs="cdn", full_html=True)
fig.show()


Previsao vendas BYD BR — 12 meses (Holt-Winters aditivo):
  Jul/2026: 3,720 un  [3,504 .. 3,936]  (faixa +-216)
  Jun/2027: 4,091 un  [3,343 .. 4,839]  (faixa +-748)
  funil abre 3.5x do 1o ao 12o mes — incerteza cresce com o horizonte


### ⑤ Recado executivo — Previsão
> - **Uma previsão sem faixa é uma opinião, não um número.** Exija sempre o
>   intervalo (ex.: 80%): a largura dele é a parte mais honesta do gráfico.
> - **A incerteza abre como um funil.** Prever o mês que vem é fácil; daqui a um
>   ano, não. Planeje capacidade de Camaçari pela borda superior/inferior, não
>   pela linha central.
> - **Modelo simples, bem calibrado, já entrega muito.** Holt-Winters (nível +
>   tendência + sazonal) captura o essencial — sofisticação a mais raramente
>   paga a conta contra uma boa leitura da faixa.


In [8]:
# ──────────────────────────────────────────────────────────────
# Exporta o resumo executivo em JSON
# ──────────────────────────────────────────────────────────────
resumo = {
    "notebook": "L4 · Time Series Analysis for Executives",
    "computed_at": pd.Timestamp.today().strftime("%Y-%m-%d"),
    "audience": "executivos nao-tecnicos",
    "format": "narrativa-primeiro (Conceito -> Intuicao -> Matematica -> Codigo -> Recado)",
    "concepts": {
        "serie_temporal": {
            "titulo": "Serie temporal — o numero com o caminho anexado",
            "definicao": "sequencia do mesmo indicador em intervalos regulares, na ordem em que ocorreu",
            "decomposicao": "y_t = tendencia (T) + sazonalidade (S) + residuo (R)",
            "exemplo_byd": {
                "serie": "PTAX mensal 2021-2026",
                "atual_R": round(float(ptax_s.iloc[-1]), 4),
                "pico_R": round(float(ptax_s.max()), 4),
                "vale_R": round(float(ptax_s.min()), 4),
                "amplitude_pct": round(float((ptax_s.max()-ptax_s.min())/ptax_s.min()*100), 1),
            },
            "insight": "o mesmo numero subindo ou caindo pede decisoes opostas; a ordem carrega a informacao",
            "pergunta_reuniao": "Esse 5,12 vem subindo ou caindo? Sem o caminho, e meia-informacao.",
        },
        "tendencia": {
            "titulo": "Tendencia — rumo estrutural, nao o ultimo ponto",
            "ferramenta": "media movel de 12 meses + reta de tendencia (slope beta1)",
            "exemplo_byd": {
                "slope_R_por_mes": round(float(beta1), 4),
                "deriva_R_por_ano": round(float(deriva_ano), 3),
                "rumo": "alta (depreciacao do real)" if beta1 > 0 else "baixa",
            },
            "insight": "media movel filtra o soluco e revela o rumo; sinal do slope = direcao, magnitude = velocidade",
            "pergunta_reuniao": "A queda do mes e soluco ou a media movel virou de inclinacao?",
        },
        "sazonalidade": {
            "titulo": "Sazonalidade — o padrao que volta todo ano",
            "metodo": "destendenciar (y/MM) e tirar media por mes do ano",
            "exemplo_byd": {
                "mes_forte": nomes[mes_forte-1], "fator_forte": round(float(fator.max()), 2),
                "mes_fraco": nomes[mes_fraco-1], "fator_fraco": round(float(fator.min()), 2),
                "spread_pct": round(float(spread), 0),
            },
            "regra_ouro": "compare YoY (mesmo mes, ano anterior), nunca contra o mes vizinho",
            "insight": "dezembro sempre bate julho — e calendario, nao merito de gestao",
            "pergunta_reuniao": "Esse +15% e gestao ou e so o pico sazonal que acontece todo ano?",
        },
        "estacionariedade": {
            "titulo": "Estacionariedade — estavel (previsivel) vs. a deriva",
            "teste": "ADF (Augmented Dickey-Fuller); p<0.05 => estacionaria",
            "exemplo_byd": {
                "adf_p_nivel": round(float(p_nivel), 3),
                "adf_p_variacao": round(float(p_dif), 3),
                "nivel_veredito": veredito(p_nivel),
                "variacao_veredito": veredito(p_dif),
            },
            "truque": "diferenciar (delta = y_t - y_{t-1}) estabiliza series que vagueiam",
            "insight": "'cambio medio dos ultimos 5 anos' nao preve nada; o nivel passeia, a variacao e estavel",
            "pergunta_reuniao": "Esse modelo rodou em serie estacionaria ou em serie a deriva?",
        },
        "autocorrelacao": {
            "titulo": "Autocorrelacao — o quanto a serie preve a si mesma",
            "formula": "rho_k = corr(y_t, y_{t-k}); banda de ruido +-2/raizT",
            "exemplo_byd": {
                "acf_lag1": round(float(acf_v[1]), 2),
                "acf_lag12": round(float(acf_v[12]), 2),
                "lags_significativos": [int(k) for k in signif],
                "banda_ruido": round(float(2/np.sqrt(len(vendas_s))), 2),
            },
            "insight": "pico no lag 12 = assinatura de sazonalidade anual; barras dentro da banda sao ruido",
            "pergunta_reuniao": "O passado recente carrega informacao sobre o proximo mes, ou e sorteio?",
        },
        "forecast": {
            "titulo": "Previsao — olhar a frente com intervalo honesto",
            "metodo": "Holt-Winters aditivo (nivel + tendencia + sazonal); faixa ~ z*sigma*raizh",
            "exemplo_byd": {
                "horizonte_meses": int(H),
                "primeiro_mes": {"data": idx_fc[0].strftime("%Y-%m"),
                                  "ponto": round(float(fc.iloc[0]), 0),
                                  "lo80": round(float(lo[0]), 0), "hi80": round(float(hi[0]), 0)},
                "ultimo_mes": {"data": idx_fc[-1].strftime("%Y-%m"),
                                "ponto": round(float(fc.iloc[-1]), 0),
                                "lo80": round(float(lo[-1]), 0), "hi80": round(float(hi[-1]), 0)},
                "funil_abre_x": round(float(banda_fc[-1]/banda_fc[0]), 1),
            },
            "insight": "a largura da faixa e a info mais honesta; incerteza abre como funil com o horizonte",
            "pergunta_reuniao": "Qual o intervalo (nao so o ponto)? Planejamos capacidade pela faixa?",
        },
    },
    "byd_context": BYD,
    "executive_phrases": [
        "Todo indicador de board e uma serie, nao um ponto — 5,12 subindo e 5,12 caindo pedem decisoes opostas.",
        "Tendencia e rumo (media movel de 12m), nao o ultimo ponto; nao confunda soluco com virada de inclinacao.",
        "Compare YoY, nunca contra o mes vizinho — dezembro sempre bate julho por calendario, nao por gestao.",
        "Pergunte se o modelo rodou em serie estacionaria; 'cambio medio de 5 anos' preve nada porque o nivel passeia.",
        "Autocorrelacao com pico no lag 12 confirma sazonalidade anual; barras dentro de +-2/raizT sao ruido.",
        "Previsao sem faixa e opiniao; exija o intervalo de 80% e planeje capacidade pela borda, nao pela linha central.",
    ],
    "palette_validated": {
        "mode":  "dark",
        "surface": "#0d1117",
        "swatches": ["#0284c7", "#dc2626", "#0d9488", "#9333ea", "#ea580c"],
        "validator": "dataviz/scripts/validate_palette.js --mode dark",
        "result": "ALL CHECKS PASS",
    },
    "visualizations": [
        "l4-01-serie-ptax.html", "l4-02-tendencia.html", "l4-03-sazonalidade.html",
        "l4-04-estacionariedade.html", "l4-05-autocorrelacao.html", "l4-06-forecast.html",
    ],
}

out_path = OUT_DIR / "l4_timeseries_executive.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(resumo, f, indent=2, ensure_ascii=False)

print("Resumo executivo salvo em:")
print(" ", out_path)
print()
print("Frases para levar a reuniao de forecast:")
for i, frase in enumerate(resumo["executive_phrases"], 1):
    print(f"  {i}. {frase}")


Resumo executivo salvo em:
  C:\Users\mathe\code_space\orchestration\value-factory\case-studies\byd-camacari-2025-2027\analise-prescritiva\outputs\learning\l4_timeseries_executive.json

Frases para levar a reuniao de forecast:
  1. Todo indicador de board e uma serie, nao um ponto — 5,12 subindo e 5,12 caindo pedem decisoes opostas.
  2. Tendencia e rumo (media movel de 12m), nao o ultimo ponto; nao confunda soluco com virada de inclinacao.
  3. Compare YoY, nunca contra o mes vizinho — dezembro sempre bate julho por calendario, nao por gestao.
  4. Pergunte se o modelo rodou em serie estacionaria; 'cambio medio de 5 anos' preve nada porque o nivel passeia.
  5. Autocorrelacao com pico no lag 12 confirma sazonalidade anual; barras dentro de +-2/raizT sao ruido.
  6. Previsao sem faixa e opiniao; exija o intervalo de 80% e planeje capacidade pela borda, nao pela linha central.


---

*L4 concluído.* Você agora domina a **dimensão do tempo**: reconhecer uma
**série** (o número com o caminho), separar **tendência** de **sazonalidade**,
checar **estacionariedade** antes de modelar, medir o eco com **autocorrelação**
e projetar o futuro com uma **faixa honesta** de previsão.

Esse vocabulário sustenta os cadernos quantitativos do case BYD: o **NB-01**
(PTAX + GARCH) modela a *volatilidade* da série temporal do câmbio; o
**NB-06/08** (Monte Carlo) simula *caminhos* futuros; o **NB-11** (backtesting)
testa previsões contra o que de fato aconteceu no tempo.

**As seis perguntas que L4 coloca em qualquer reunião de dados:**

1. *Esse número vem subindo ou caindo?* (série)
2. *É soluço ou o rumo virou?* (tendência)
3. *Isso é gestão ou é sazonalidade?* (sazonalidade)
4. *O modelo rodou em série estável?* (estacionariedade)
5. *O passado carrega informação sobre o futuro?* (autocorrelação)
6. *Qual o intervalo, não só o ponto?* (forecast)

**Próximo nível:** `L5 — Classificação e Árvores de Decisão` — quando o que
você quer prever não é um número no tempo, mas uma **categoria** (aprovar/negar,
churn/reter, fornecedor de risco alto/baixo).
